# Lab 02 — Evaluando un Agente de Arquitectura Cloud-Native

**Objetivo:** Aprender a evaluar la *calidad* de las respuestas de un agente especializado, no solo si responde.

En este lab construimos **ArchitectAI**, un agente experto en patrones de arquitectura cloud-native, y lo evaluamos con tres dimensiones:

| Evaluador | Tipo | ¿Qué mide? |
|---|---|---|
| **Accuracy** | LLM-as-Judge | ¿Cubre los temas esperados? (score 0-10) |
| **Hallucination** | LLM-as-Judge | ¿Hace afirmaciones falsas o peligrosas? |
| **Response Quality** | Rule-Based | Longitud, estructura markdown, patrones citados, trade-offs |

También medimos métricas de rendimiento del agente: **TTFT**, **TTC** y **coste** por pregunta.

## ¿Por qué evaluar un agente de Q&A?

Un agente puede *responder* a cualquier pregunta sin que eso signifique que *responde bien*. En dominios técnicos como arquitectura de software:

- Una respuesta **incorrecta** puede llevar a decisiones de diseño costosas y difíciles de revertir.
- Una respuesta **incompleta** puede omitir trade-offs críticos que el desarrollador necesita conocer.
- Una respuesta con **alucinaciones** puede citar patrones inexistentes o afirmar que algo "es simple" cuando no lo es.

Los evals nos permiten cuantificar estas dimensiones y comparar modelos de forma objetiva antes de ponerlos en producción.

In [ ]:
# Setup — imports, API key, paths
import anthropic
import json
import os
import re
import time
from datetime import datetime, timezone
from pathlib import Path

from dotenv import load_dotenv

load_dotenv()


def _find_repo_root(start: Path = Path.cwd()) -> Path:
    """Walk up from start until we find config/rules.yaml (repo root marker)."""
    current = start.resolve()
    for parent in [current, *current.parents]:
        if (parent / "config" / "rules.yaml").exists():
            return parent
    return current  # fallback: cwd


REPO_ROOT = _find_repo_root()
RESULTS_DIR = REPO_ROOT / "results"
RESULTS_DIR.mkdir(exist_ok=True)

api_key = os.environ.get("ANTHROPIC_API_KEY")
if not api_key:
    api_key = input("Introduce tu ANTHROPIC_API_KEY: ").strip()

client = anthropic.Anthropic(api_key=api_key)
print(f"Cliente Anthropic configurado. Repo root: {REPO_ROOT}")

## Definición del Agente ArchitectAI

ArchitectAI es un agente especializado con un system prompt diseñado para:
1. Acotar el dominio (patrones cloud-native específicos).
2. Fomentar respuestas matizadas — mencionar trade-offs, evitar absolutismos.
3. Usar terminología canónica ("Saga pattern", "Circuit Breaker").

Usamos **streaming** para capturar métricas de rendimiento (TTFT, TTC, tokens/segundo).

In [ ]:
# ArchitectAI system prompt + ask_architect() con streaming y métricas de performance

ARCHITECT_SYSTEM_PROMPT = """Eres ArchitectAI, un experto en arquitectura de software cloud-native \
con 15 años de experiencia diseñando sistemas distribuidos a escala.

Tu dominio de conocimiento incluye:
- Patrones de diseño: CQRS, Saga (choreography y orchestration), Circuit Breaker, \
  API Gateway, Sidecar, Service Mesh
- Metodologías: 12-factor app, Event-Driven Architecture, Domain-Driven Design
- Operaciones: Blue-Green Deployment, Canary Release, Chaos Engineering
- Trade-offs: cuándo aplicar cada patrón y cuándo NO aplicarlo

Principios que guían tus respuestas:
1. Siempre menciona trade-offs y casos donde el patrón NO aplica.
2. Nunca hagas afirmaciones absolutas como "siempre usa X" o "X es simple".
3. Cita patrones por su nombre canónico (ej: "Saga pattern", "Circuit Breaker").
4. Estructura tus respuestas con secciones claras cuando la pregunta lo amerita.
5. Si la pregunta implica una decisión de diseño, ofrece criterios de decisión concretos.

Responde siempre en el mismo idioma en que se hace la pregunta."""

# Pricing (USD por millón de tokens: input, output)
PRICING = {
    "claude-haiku-4-5-20251001": (0.80, 4.00),
    "claude-sonnet-4-6": (3.00, 15.00),
    "claude-opus-4-7": (15.00, 75.00),
}


def cost_usd(model: str, tin: int, tout: int) -> float:
    pin, pout = PRICING.get(model, (3.00, 15.00))
    return round((tin * pin + tout * pout) / 1_000_000, 6)


def ask_architect(question: str, model: str = "claude-sonnet-4-6") -> dict:
    """Send a question to ArchitectAI with streaming + performance metrics."""
    ttft = None
    start = time.perf_counter()
    with client.messages.stream(
        model=model,
        max_tokens=1024,
        system=ARCHITECT_SYSTEM_PROMPT,
        messages=[{"role": "user", "content": question}],
    ) as stream:
        for event in stream:
            if ttft is None and event.type == "content_block_start":
                ttft = time.perf_counter() - start
        final_msg = stream.get_final_message()
    ttc = time.perf_counter() - start
    answer = "".join(b.text for b in final_msg.content if hasattr(b, "text"))
    input_tokens = final_msg.usage.input_tokens
    output_tokens = final_msg.usage.output_tokens
    ttc_ms = round(ttc * 1000, 1)
    ttft_ms = round((ttft or ttc) * 1000, 1)
    otps = round(output_tokens / max((ttc - (ttft or ttc)), 1e-6), 2)
    return {
        "question": question,
        "answer": answer,
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
        "total_tokens": input_tokens + output_tokens,
        "ttft_ms": ttft_ms,
        "ttc_ms": ttc_ms,
        "otps": otps,
        "cost_usd": cost_usd(model, input_tokens, output_tokens),
    }

print("ArchitectAI definido correctamente.")

## Golden Dataset — 10 Preguntas

El dataset cubre los patrones principales del dominio cloud-native. Cada entrada incluye:
- **`expected_themes`**: conceptos que una buena respuesta debe mencionar.
- **`forbidden_claims`**: afirmaciones que indican alucinación o consejo peligroso.

Este dataset es la fuente de verdad del lab — define qué significa "responder bien".

In [ ]:
# Golden dataset completo — 10 preguntas sobre patrones cloud-native
GOLDEN_DATASET = [
    {
        "id": "q01",
        "topic": "CQRS",
        "question": "¿Cuándo usar CQRS vs una arquitectura tradicional CRUD?",
        "expected_themes": [
            "read/write separation",
            "scalability",
            "complexity tradeoff",
            "event sourcing",
            "bounded context",
        ],
        "forbidden_claims": [
            "always use CQRS",
            "CQRS is simple",
            "CQRS replaces CRUD entirely",
        ],
    },
    {
        "id": "q02",
        "topic": "Saga pattern",
        "question": "¿Qué patrón usar para transacciones distribuidas en microservicios?",
        "expected_themes": [
            "saga pattern",
            "choreography",
            "orchestration",
            "compensating transactions",
            "eventual consistency",
        ],
        "forbidden_claims": [
            "use 2PC in microservices",
            "distributed transactions are simple",
            "ACID guarantees in microservices",
        ],
    },
    {
        "id": "q03",
        "topic": "Circuit Breaker",
        "question": "¿Cómo evitar fallos en cascada cuando un servicio dependiente está caído?",
        "expected_themes": [
            "circuit breaker",
            "fallback",
            "half-open state",
            "timeout",
            "bulkhead",
        ],
        "forbidden_claims": [
            "just add retries",
            "circuit breaker is only for HTTP",
            "circuit breaker eliminates all failures",
        ],
    },
    {
        "id": "q04",
        "topic": "API Gateway",
        "question": "¿Cuál es el rol de un API Gateway en una arquitectura de microservicios?",
        "expected_themes": [
            "single entry point",
            "routing",
            "authentication",
            "rate limiting",
            "aggregation",
        ],
        "forbidden_claims": [
            "API Gateway is a single point of failure without mitigation",
            "API Gateway handles all business logic",
            "you must use one API Gateway per service",
        ],
    },
    {
        "id": "q05",
        "topic": "Event-Driven Architecture",
        "question": "¿Qué ventajas e inconvenientes tiene la arquitectura orientada a eventos?",
        "expected_themes": [
            "loose coupling",
            "asynchronous",
            "eventual consistency",
            "event broker",
            "debugging complexity",
        ],
        "forbidden_claims": [
            "event-driven is always better than REST",
            "events guarantee exactly-once delivery trivially",
            "event-driven eliminates all coupling",
        ],
    },
    {
        "id": "q06",
        "topic": "12-factor app",
        "question": "¿Qué significa que una aplicación sea '12-factor' y por qué importa en cloud-native?",
        "expected_themes": [
            "config in environment",
            "stateless processes",
            "port binding",
            "disposability",
            "dev/prod parity",
        ],
        "forbidden_claims": [
            "12-factor apps have no state",
            "12-factor is only for containers",
            "12-factor guarantees scalability automatically",
        ],
    },
    {
        "id": "q07",
        "topic": "Service Mesh y Sidecar",
        "question": "¿Cuándo justifica la complejidad operacional de un service mesh?",
        "expected_themes": [
            "sidecar proxy",
            "observability",
            "mutual TLS",
            "traffic management",
            "operational overhead",
        ],
        "forbidden_claims": [
            "always use a service mesh",
            "service mesh has no performance impact",
            "service mesh replaces API Gateway",
        ],
    },
    {
        "id": "q08",
        "topic": "Blue-Green Deployment",
        "question": "¿Cuál es la diferencia entre blue-green deployment y canary release?",
        "expected_themes": [
            "zero downtime",
            "traffic switching",
            "rollback",
            "canary percentage",
            "risk mitigation",
        ],
        "forbidden_claims": [
            "blue-green and canary are the same",
            "blue-green has no cost implications",
            "canary release requires no monitoring",
        ],
    },
    {
        "id": "q09",
        "topic": "Chaos Engineering",
        "question": "¿Qué es chaos engineering y cómo se integra en el SDLC?",
        "expected_themes": [
            "hypothesis",
            "steady state",
            "blast radius",
            "fault injection",
            "resilience verification",
        ],
        "forbidden_claims": [
            "run chaos in production without preparation",
            "chaos engineering is only for Netflix",
            "chaos engineering replaces load testing",
        ],
    },
    {
        "id": "q10",
        "topic": "Strangler Fig Pattern",
        "question": "¿Cómo migrar un monolito a microservicios sin big-bang rewrite?",
        "expected_themes": [
            "strangler fig",
            "incremental migration",
            "facade",
            "risk reduction",
            "feature parity",
        ],
        "forbidden_claims": [
            "rewrite everything at once",
            "microservices are always better than monoliths",
            "strangler fig has no overhead",
        ],
    },
]

print(f"Golden dataset cargado: {len(GOLDEN_DATASET)} preguntas.")

## Prueba Rápida con una Pregunta

Antes de ejecutar el dataset completo, verificamos que el agente responde correctamente con la primera pregunta.

In [ ]:
# Prueba rápida con la primera pregunta del dataset
sample = ask_architect(GOLDEN_DATASET[0]["question"])

print(f"Pregunta: {GOLDEN_DATASET[0]['question']}")
print(f"\nRespuesta (primeros 500 chars):\n{sample['answer'][:500]}...")
print(f"\n--- Métricas de rendimiento ---")
print(f"  TTFT:          {sample['ttft_ms']} ms")
print(f"  TTC:           {sample['ttc_ms']} ms")
print(f"  Output tokens/s: {sample['otps']}")
print(f"  Tokens:        {sample['input_tokens']} in / {sample['output_tokens']} out")
print(f"  Coste:         ${sample['cost_usd']}")

## Los Tres Evaluadores

Implementamos cada evaluador en su propia celda para que sea fácil inspeccionarlos y modificarlos de forma independiente.

- **Accuracy Judge** y **Hallucination Judge**: llaman al API de Anthropic (LLM-as-Judge).
- **Response Quality**: sin llamadas LLM — solo reglas deterministas.

In [ ]:
# Evaluador 1: Accuracy (LLM-as-Judge)

ACCURACY_JUDGE_PROMPT = """Eres un evaluador experto en arquitectura cloud-native. \
Tu tarea es evaluar si una respuesta cubre adecuadamente los temas esperados.

Pregunta: {question}

Respuesta del agente:
{answer}

Temas esperados (al menos 2/3 deben estar presentes para aprobar):
{expected_themes}

Evalúa en una escala de 0-10:
- 9-10: Todos los temas cubiertos con profundidad y precisión técnica.
- 7-8: La mayoría de temas cubiertos con suficiente detalle.
- 5-6: Algunos temas presentes pero superficiales o incompletos.
- 3-4: Pocos temas cubiertos; la respuesta no es útil para tomar decisiones.
- 0-2: Los temas esperados están ausentes o la respuesta es incorrecta.

Responde ÚNICAMENTE con un JSON válido con este formato exacto:
{{"score": <número entre 0 y 10>, "justification": "<máximo 2 oraciones>"}}"""


def run_llm_judge(prompt: str, max_tokens: int = 256) -> dict | None:
    """Wrapper for LLM judge calls with error handling."""
    try:
        resp = client.messages.create(
            model="claude-sonnet-4-6",
            max_tokens=max_tokens,
            messages=[{"role": "user", "content": prompt}],
        )
        return json.loads(resp.content[0].text)
    except (anthropic.APIStatusError, json.JSONDecodeError) as e:
        print(f"Judge error: {e}")
        return None


def check_accuracy(question: str, answer: str, expected_themes: list) -> dict:
    prompt = ACCURACY_JUDGE_PROMPT.format(
        question=question,
        answer=answer,
        expected_themes=", ".join(expected_themes),
    )
    result = run_llm_judge(prompt, max_tokens=256)
    if result is None:
        return {"score": None, "justification": None, "error": "judge call failed"}
    return {"score": result.get("score"), "justification": result.get("justification")}


print("check_accuracy() definida.")

In [ ]:
# Evaluador 2: Hallucination (LLM-as-Judge)

HALLUCINATION_JUDGE_PROMPT = """Eres un auditor técnico especializado en detectar \
afirmaciones falsas o peligrosas en respuestas sobre arquitectura cloud-native.

Pregunta: {question}

Respuesta del agente:
{answer}

Afirmaciones prohibidas conocidas (presencia directa = alucinación confirmada):
{forbidden_claims}

Tu tarea:
1. Comprueba si alguna forbidden_claim aparece literalmente o de forma parafraseada.
2. Identifica cualquier otra afirmación técnicamente incorrecta en el dominio \
   cloud-native, aunque no esté en la lista de forbidden_claims.

Responde ÚNICAMENTE con un JSON válido:
{{
  "has_hallucination": <true o false>,
  "forbidden_claims_found": [<lista de claims encontradas, puede estar vacía>],
  "other_issues": [<lista de otras afirmaciones falsas detectadas, puede estar vacía>],
  "severity": "<none | low | medium | high>"
}}"""


def check_hallucination(question: str, answer: str, forbidden_claims: list) -> dict:
    prompt = HALLUCINATION_JUDGE_PROMPT.format(
        question=question,
        answer=answer,
        forbidden_claims="\n".join(f"- {c}" for c in forbidden_claims),
    )
    result = run_llm_judge(prompt, max_tokens=512)
    if result is None:
        return {
            "has_hallucination": None,
            "forbidden_claims_found": [],
            "other_issues": [],
            "severity": "unknown",
            "error": "judge call failed",
        }
    return {
        "has_hallucination": result.get("has_hallucination"),
        "forbidden_claims_found": result.get("forbidden_claims_found", []),
        "other_issues": result.get("other_issues", []),
        "severity": result.get("severity", "none"),
    }


print("check_hallucination() definida.")

In [ ]:
# Evaluador 3: Response Quality (Rule-Based — sin llamadas LLM)

def check_response_quality(answer: str, expected_themes: list) -> dict:
    """Rule-based quality checker. No LLM calls required."""
    score = 0
    details = {}

    # Rule 1: Minimum length (200-1500 words is the target range)
    word_count = len(answer.split())
    details["word_count"] = word_count
    if word_count >= 200:
        score += 3
    elif word_count >= 100:
        score += 1

    # Rule 2: Uses markdown structure (headers or bullet points)
    has_structure = bool(re.search(r"(^#{1,3} |\n#{1,3} |\n- |\n\* |\n\d+\. )", answer))
    details["has_structure"] = has_structure
    if has_structure:
        score += 2

    # Rule 3: References known patterns by canonical name
    pattern_keywords = [
        "CQRS", "Saga", "Circuit Breaker", "API Gateway", "Sidecar",
        "Service Mesh", "12-factor", "Blue-Green", "Canary", "Strangler",
        "Event-Driven", "Chaos Engineering", "Bulkhead", "Compensating",
    ]
    found_patterns = [kw for kw in pattern_keywords if kw.lower() in answer.lower()]
    details["patterns_referenced"] = found_patterns
    score += min(len(found_patterns), 3)

    # Rule 4: Mentions trade-offs (positive signal)
    tradeoff_keywords = ["tradeoff", "trade-off", "ventaja", "inconveniente",
                         "cuando no", "desventaja", "complejidad", "overhead"]
    mentions_tradeoffs = any(kw in answer.lower() for kw in tradeoff_keywords)
    details["mentions_tradeoffs"] = mentions_tradeoffs
    if mentions_tradeoffs:
        score += 2

    return {"score": min(score, 10), "details": details}


print("check_response_quality() definida.")

## Pipeline Completo de Evaluación

Ejecutamos las 10 preguntas del dataset. Para cada una:
1. El agente genera una respuesta (con métricas de rendimiento).
2. Los tres evaluadores analizan la respuesta.
3. Guardamos todos los resultados en una lista estructurada.

**Tiempo estimado:** ~2-3 minutos con `claude-sonnet-4-6`.

In [ ]:
# Pipeline completo — agente + 3 evaluadores para las 10 preguntas
MODEL = os.environ.get("MODEL_ID", "claude-sonnet-4-6")
print(f"Modelo del agente: {MODEL}")
print("=" * 60)

eval_results = []
for item in GOLDEN_DATASET:
    print(f"  [{item['id']}] {item['topic']}...")

    # Step 1: Generate response from ArchitectAI (streaming + perf metrics)
    agent_out = ask_architect(item["question"], model=MODEL)

    # Step 2: Run evaluators
    accuracy = check_accuracy(item["question"], agent_out["answer"], item["expected_themes"])
    hallucination = check_hallucination(item["question"], agent_out["answer"], item["forbidden_claims"])
    quality = check_response_quality(agent_out["answer"], item["expected_themes"])

    eval_results.append({
        "id": item["id"],
        "topic": item["topic"],
        "question": item["question"],
        "answer": agent_out["answer"],
        "performance": {
            "ttft_ms": agent_out["ttft_ms"],
            "ttc_ms": agent_out["ttc_ms"],
            "otps": agent_out["otps"],
            "input_tokens": agent_out["input_tokens"],
            "output_tokens": agent_out["output_tokens"],
            "total_tokens": agent_out["total_tokens"],
            "cost_usd": agent_out["cost_usd"],
        },
        "accuracy": accuracy,
        "hallucination": hallucination,
        "quality": quality,
    })

print(f"\nEvaluacion completada: {len(eval_results)}/10 preguntas procesadas.")

## Métricas Agregadas

Calculamos los KPIs del run: calidad promedio, tasa de alucinaciones, coste total y rendimiento medio.

In [ ]:
# Compute summary con todas las métricas de calidad y rendimiento
n = len(eval_results)

accuracy_scores = [r["accuracy"]["score"] for r in eval_results if r["accuracy"].get("score") is not None]
hallucination_flags = [r["hallucination"]["has_hallucination"] for r in eval_results if r["hallucination"].get("has_hallucination") is not None]
quality_scores = [r["quality"]["score"] for r in eval_results]
total_in = sum(r["performance"]["input_tokens"] for r in eval_results)
total_out = sum(r["performance"]["output_tokens"] for r in eval_results)

summary = {
    "questions_evaluated": n,
    "accuracy_score": round(sum(accuracy_scores) / len(accuracy_scores), 2) if accuracy_scores else None,
    "hallucination_rate": round(sum(1 for h in hallucination_flags if h) / len(hallucination_flags), 4) if hallucination_flags else None,
    "response_quality": round(sum(quality_scores) / n, 2),
    "avg_tokens": round(total_out / n, 1),
    "total_input_tokens": total_in,
    "total_output_tokens": total_out,
    "total_cost_usd": round(sum(r["performance"]["cost_usd"] for r in eval_results), 6),
    "avg_ttft_ms": round(sum(r["performance"]["ttft_ms"] for r in eval_results) / n, 1),
    "avg_ttc_ms": round(sum(r["performance"]["ttc_ms"] for r in eval_results) / n, 1),
}

print(json.dumps(summary, indent=2))

## Análisis por Pregunta

Tabla con los resultados detallados de cada pregunta.

In [ ]:
# Tabla por pregunta: id, topic, accuracy, hallucination, quality, ttft, cost
header = f"{'ID':<5} {'Topic':<28} {'Acc':>4} {'Halluc':>7} {'Qual':>5} {'TTFT(ms)':>9} {'Cost($)':>9}"
print(header)
print("-" * len(header))

for r in eval_results:
    acc = r["accuracy"].get("score", "ERR")
    acc_str = f"{acc:.1f}" if isinstance(acc, (int, float)) else str(acc)
    hall = r["hallucination"].get("has_hallucination")
    hall_str = "YES" if hall else ("no" if hall is False else "ERR")
    qual = r["quality"]["score"]
    ttft = r["performance"]["ttft_ms"]
    cost = r["performance"]["cost_usd"]
    print(f"{r['id']:<5} {r['topic']:<28} {acc_str:>4} {hall_str:>7} {qual:>5} {ttft:>9.1f} {cost:>9.6f}")

## Casos de Fallo — Análisis Cualitativo

Filtramos las respuestas problemáticas para entender **qué falló** y cómo mejorar el system prompt o el dataset.

In [ ]:
# Filtrar preguntas con accuracy < 6 o has_hallucination == True
failures = [
    r for r in eval_results
    if (r["accuracy"].get("score") is not None and r["accuracy"]["score"] < 6)
    or r["hallucination"].get("has_hallucination") is True
]

if not failures:
    print("No se encontraron casos de fallo (accuracy < 6 o alucinacion detectada).")
else:
    print(f"{len(failures)} caso(s) de fallo encontrado(s):\n")
    for r in failures:
        print(f"--- {r['id']} ({r['topic']}) ---")
        print(f"Pregunta:     {r['question']}")
        print(f"Accuracy:     {r['accuracy'].get('score')} — {r['accuracy'].get('justification')}")
        print(f"Alucinacion:  {r['hallucination'].get('has_hallucination')} (severity: {r['hallucination'].get('severity')})")
        if r["hallucination"].get("forbidden_claims_found"):
            print(f"Claims:       {r['hallucination']['forbidden_claims_found']}")
        print(f"Respuesta:\n{r['answer'][:600]}...\n")

In [ ]:
# Exportar JSON a results/YYYY-MM-DD_HH-MM_lab02_<model>.json
model_slug = MODEL.replace("/", "-")
ts = datetime.now().strftime("%Y-%m-%d_%H-%M")
output_file = RESULTS_DIR / f"{ts}_lab02_{model_slug}.json"

run_output = {
    "run_id": datetime.now(timezone.utc).isoformat(),
    "lab": "lab02",
    "model": MODEL,
    "summary": summary,
    "details": eval_results,
}

with open(output_file, "w", encoding="utf-8") as f:
    json.dump(run_output, f, indent=2, ensure_ascii=False)

print(f"Resultados exportados a: {output_file}")

## Reflexión — ¿Cómo escalar estos evals?

### Preguntas para debate

**1. Comparación de modelos**
¿Qué diferencias esperas al cambiar `claude-sonnet-4-6` por `claude-haiku-4-5`? ¿En qué métricas?
- `haiku` es ~4× más rápido y ~10× más barato. ¿Sacrifica calidad de razonamiento en dominios técnicos?
- Pista: mira los `accuracy_score` y `hallucination_rate` — ahí suele aparecer la diferencia más clara.

**2. Limitaciones del golden dataset**
Los `expected_themes` son palabras clave. ¿Qué tipos de respuestas correctas podrían obtener baja puntuación?
- Una respuesta que use sinónimos perfectamente válidos pero no los términos exactos.
- ¿Cómo mejorarías el accuracy judge para manejar esto?

**3. Escalar a 100 preguntas**
Con 10 preguntas el coste es < $0.05. Con 100 preguntas y runs diarios en CI:
- ¿Qué estrategia de sampling usarías para mantener cobertura sin escalar el coste linealmente?
- ¿Cuándo tiene sentido usar `haiku` como juez en lugar de `sonnet`?
- ¿Cómo detectarías regresiones de calidad entre versiones del system prompt?

**4. Más allá del Q&A**
Este lab evalúa respuestas textuales. ¿Cómo adaptarías este framework para evaluar un agente que **ejecuta acciones** (crea infraestructura, llama APIs, escribe código)?